<a href="https://colab.research.google.com/github/2xsec/2xsec.github.io/blob/master/04_Model_Compression_Pipeline_difficult_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Model Compression Pipeline 실습**

이번 실습에서는 실제 모델 경량화가 한 가지 기법만으로 끝나지 않는다는 점을 확인합니다.  
하나의 Student 모델에 **Knowledge Distillation(KD)**, **Weight Decomposition**, **Pruning**, **Quantization**을 순차적으로 적용하고, 단계마다 정확도, 모델 크기, 파라미터 수, 추론 시간을 비교합니다.

이번 실습에서 배워갈 내용은 다음과 같습니다.

1. 큰 Teacher 모델의 지식을 작은 Student 모델로 전달하는 방법
2. `tensorly.parafac()` 기반 CP Weight Decomposition으로 큰 Linear layer를 두 개의 작은 layer로 나누는 방법
3. Pruning으로 중요도가 낮은 weight를 제거하고 sparsity를 관찰하는 방법
4. Dynamic Quantization으로 배포용 CPU 모델의 저장 크기를 줄이는 방법
5. 여러 경량화 기법을 누적 적용할 때 성능이 어떻게 변하는지 비교하는 방법

# 0. Colab 환경설정
- Colab에서 GPU를 사용할 수 있도록 설정합니다.
    - 런타임 > 런타임 유형 변경 > Python 3 및 T4 GPU 선택
    - colab에서 Google Drive에 접근할 수 있도록 설정
- 마지막 Quantization 평가는 CPU에서 수행합니다. PyTorch Dynamic Quantization은 주로 CPU inference에 적용되기 때문입니다.

## 0-1. Setup
필요한 package를 import하고 실험 재현성을 위해 seed를 고정합니다.

- 여러 경량화 기법을 한 노트북에서 비교하려면 동일한 seed와 동일한 평가 함수가 중요합니다.
- GPU는 학습에 사용하고, CPU는 quantized model 평가에 사용합니다.

In [ ]:
! pip install tensorly==0.8.1

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

print("[현재 파일 위치]")
!pwd
print("[현재 디렉토리의 파일 확인]")
!ls

In [ ]:
# 본인 환경에 맞게 경로를 수정하여 사용하세요.
%cd /content/drive/MyDrive/day3
!pwd

In [ ]:
# copy: 모델을 복사하여 원본 모델을 보존할 때 사용합니다.
# os: Colab/로컬 경로 확인 등 일반 파일 경로 처리에 사용합니다.
# time: inference latency를 측정할 때 사용합니다.
import copy
import os
import random
import sys
import time
from pathlib import Path

# numpy: 평균 latency 계산과 seed 고정에 사용합니다.
import numpy as np

# torch: 모델 정의, 학습, pruning, quantization에 필요한 핵심 라이브러리입니다.
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils.prune as prune
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm import tqdm

# tensorly의 CP decomposition을 사용합니다.
from tensorly.decomposition import parafac
import tensorly as tl

# 평가/출력 함수는 utils.py에서 불러옵니다.
from utils import (
    benchmark_latency,
    count_parameters,
    evaluate_accuracy,
    format_count,
    format_float,
    get_model_size_mb,
    record_result as utils_record_result,
)

In [ ]:
# 실습 결과 재현성을 위해 seed를 고정합니다.
seed = 2026
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# deterministic=True는 같은 입력에 대해 가능한 한 같은 결과가 나오도록 합니다.
# benchmark=False는 입력 크기에 따라 알고리즘을 바꾸는 최적화를 꺼서 재현성을 높입니다.
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# tensorly가 PyTorch Tensor를 backend로 사용하도록 설정합니다.
# 이 설정이 있어야 parafac 결과를 PyTorch Tensor처럼 다룰 수 있습니다.
tl.set_backend('pytorch')

# 학습은 GPU가 있으면 GPU에서 수행하고, quantization 및 latency 비교는 CPU 기준으로 수행합니다.
train_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cpu_device = torch.device("cpu")
torch.backends.quantized.engine = "fbgemm"
print(f"Train device: {train_device}")

## 0-2. 실습 관련 하이퍼파라미터 설정

- 경량화 기법의 상대적 변화를 보기 위해서는 모든 단계에서 같은 데이터 수와 평가 조건을 유지해야 합니다.
- epoch를 늘리면 정확도는 안정될 수 있지만 실습 시간이 길어지고, 작은 subset에서는 overfitting이 생길 수 있습니다.
- KD가 Student 단독 학습보다 항상 좋아지는 것은 아닙니다. 특히 MNIST처럼 쉬운 task이고 Teacher/Student 성능 차이가 작으면 KD의 이득이 작거나 없을 수 있습니다.
- 이번 실습에서는 이 현상까지 함께 관찰하면서, 경량화 기법의 효과가 task/model 조건에 따라 달라진다는 점을 확인합니다.

In [ ]:
NUM_TRAIN_SAMPLES = 12000
NUM_TEST_SAMPLES = 3000
TEACHER_EPOCHS = 3
STUDENT_EPOCHS = 3
FINETUNE_EPOCHS = 2

# KD 관련 hyperparameter입니다.
# KD_ALPHA가 클수록 Teacher soft label을 강하게 따르고, 작을수록 실제 label Cross Entropy를 더 강하게 따릅니다.
KD_ALPHA = 0.3

# KD_TEMPERATURE는 Teacher 확률 분포를 얼마나 부드럽게 만들지 정합니다.
KD_TEMPERATURE = 2.0

# KD 학습에서 learning rate가 너무 크면 Teacher soft label 쪽으로 흔들리며 성능이 떨어질 수 있습니다.
# Student 단독 학습보다 약간 작은 learning rate를 사용합니다.
KD_LR = 0.0007

# batch size가 너무 작으면 latency 측정이 불안정하고, 너무 크면 Colab 메모리 부담이 커집니다.
BATCH_SIZE = 128

# 1. Dataset 준비
MNIST 데이터를 불러오고 train/test loader를 만듭니다.

- 모든 모델을 같은 data loader로 평가해야 경량화 전후 결과를 공정하게 비교할 수 있습니다.
- 경량화 실습에서는 데이터 수를 줄여도 비교 흐름을 충분히 관찰할 수 있습니다.

In [ ]:
# MNIST 이미지를 Tensor로 바꾸고, MNIST의 평균/표준편차로 정규화합니다.
# 정규화는 학습을 더 안정적으로 만들어 줍니다.
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

# train=True는 학습 데이터, train=False는 테스트 데이터를 의미합니다.
train_dataset_full = datasets.MNIST(root="/content/data", train=True, download=True, transform=transform)
test_dataset_full = datasets.MNIST(root="/content/data", train=False, download=True, transform=transform)

# 간단한 실습을 목표로 하기 때문에 시간관계상 앞쪽 일부 sample만 사용합니다.
# 실전에서는 Subset을 제거하고 전체 데이터를 사용하는 것이 일반적입니다.
train_indices = list(range(NUM_TRAIN_SAMPLES))
test_indices = list(range(NUM_TEST_SAMPLES))

train_dataset = Subset(train_dataset_full, train_indices)
test_dataset = Subset(test_dataset_full, test_indices)

# pin_memory=True는 GPU로 데이터를 옮길 때 약간의 속도 이점을 줄 수 있습니다.
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples : {len(test_dataset)}")

# 2. Teacher 모델 & Student 모델 정의
Teacher는 상대적으로 큰 CNN, Student는 더 작은 CNN으로 정의합니다.

- KD는 큰 모델의 출력 분포를 작은 모델이 따라가도록 만드는 방법입니다.
- 이후 decomposition을 적용하기 위해 Student의 `fc1` layer를 명확히 분리해둡니다.

## 2-1. Teacher 모델 정의

In [ ]:
class TeacherCNN(nn.Module):
    """Student에게 지식을 전달할 비교적 큰 CNN입니다."""
    def __init__(self):
        super().__init__()

        # feature extractor: 이미지에서 점차 추상적인 특징을 뽑습니다.
        # Teacher는 Student보다 채널 수가 많아서 표현력이 더 큽니다.
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        # MNIST 28x28 이미지는 pooling 두 번 후 7x7 feature map이 됩니다.
        ##### [실습] 최종적으로 10개 숫자 class logit을 출력합니다. #####
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, ??),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

teacher_model = TeacherCNN().to(train_device)

## 2-2. Student 모델 정의

In [ ]:
class StudentCNN(nn.Module):
    """실제로 경량화를 누적 적용할 작은 CNN입니다."""
    def __init__(self, hidden_dim=128):
        super().__init__()

        # Student는 Teacher보다 채널 수가 작아 파라미터와 연산량이 적습니다.
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()

        # fc1은 Student에서 가장 큰 weight matrix입니다.
        # 뒤에서 tensorly.parafac으로 이 layer를 CP 분해합니다.
        self.fc1 = nn.Linear(32 * 7 * 7, hidden_dim)
        ##### [실습] 최종적으로 10개 숫자 class logit을 출력합니다. #####
        self.fc2 = nn.Linear(hidden_dim, ??)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

student_model = StudentCNN().to(train_device)

# 3. 공통 학습/평가 함수 정의
학습 함수는 노트북에서 직접 정의하고, 모델 크기/파라미터 수/latency/결과 출력 함수는 `utils.py`에서 불러와 사용합니다.


- 경량화는 정확도만 보면 부족합니다. 크기와 속도도 함께 봐야 합니다.
- Latency는 Colab CPU 상태에 따라 흔들릴 수 있으므로 `utils.py`의 `benchmark_latency()`는 warmup 이후 여러 batch의 중앙값으로 측정합니다.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """일반 supervised learning 한 epoch를 수행합니다."""

    ###### [실습] 모델을 학습 모드(train)로 설정합니다. ######
    model.??()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Train", leave=False):
        ##### [실습] 입력 데이터와 정답을 연산이 수행될 장치(CPU/GPU)로 이동합니다. #####
        images, labels = ??, ??

        ##### [실습] 입력 데이터를 모델에 전달하여 예측 결과를 계산합니다. #####
        outputs = ??

        ##### [실습] 예측 결과와 정답을 이용하여 손실(loss)을 계산합니다. #####
        loss = ??

        ##### [실습] 이전 step에서 계산된 gradient를 초기화합니다. #####
        ??

        ##### [실습] 손실값을 기반으로 gradient를 계산합니다. #####
        ??

        ##### [실습] 계산된 gradient를 이용하여 모델의 파라미터를 업데이트합니다. #####
        ??

        ##### [실습] 현재 배치의 loss를 누적하여 epoch 평균 loss를 계산합니다. #####
        ??

        ##### [실습] 예측 결과에서 가장 높은 점수를 가진 클래스를 선택합니다. #####
        ??

        ##### [실습] 올바르게 예측한 샘플 수를 누적합니다. #####
        ??

        ##### [실습] 처리한 전체 샘플 수를 누적합니다. #####
        ??

    return total_loss / total, 100.0 * correct / total


def evaluate(model, loader, device):
    """utils.py의 evaluate_accuracy를 이 노트북에서 쓰기 쉬운 이름으로 감싼 함수입니다."""
    # evaluate accuracy를 반환합니다.
    return evaluate_accuracy(model, loader, device)

results = []

def record_result(stage, model, device, note=""):
    """utils.py의 record_result에 현재 노트북의 test_loader/cpu_device/results를 연결합니다."""
    results[:] = [row for row in results if row["stage"] != stage]
    return utils_record_result(
        stage=stage,
        model=model,
        eval_loader=test_loader,
        device=device,
        cpu_device=cpu_device,
        results=results,
        note=note,
    )

# 4. Baseline Teacher 학습
Teacher 모델을 먼저 학습합니다. Teacher는 이후 Student가 따라갈 부드러운 확률 분포를 제공합니다.

- Teacher의 정확도가 높을수록 KD로 전달할 정보가 좋아집니다.
- 실습 시간상 짧게 학습하지만, 실제 프로젝트에서는 충분히 학습된 Teacher를 사용합니다.

In [ ]:
# 옵티마이저는 Adam 옵티마이저를 사용해줍니다.
teacher_optimizer = optim.Adam(teacher_model.parameters(), lr=0.001)
##### [실습] Teacher는 일반 Cross Entropy Loss로 먼저 학습합니다. #####
ce_loss = nn.??()

for epoch in range(1, TEACHER_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(teacher_model, train_loader, teacher_optimizer, ce_loss, train_device)
    test_acc = evaluate(teacher_model, test_loader, train_device)
    print(f"Epoch {epoch}/{TEACHER_EPOCHS} | loss={train_loss:.4f} | train_acc={train_acc:.2f}% | test_acc={test_acc:.2f}%")

# Teacher는 크지만 정확도가 높아 Student에게 soft target을 제공하는 역할을 합니다.
_ = record_result("Teacher Model", teacher_model, train_device, "KD를 위한 기준 모델(Teacher 모델)입니다.")

# 5. Student 모델 Knowledge Distillation 적용
Teacher의 soft label과 실제 label을 함께 사용해 Student를 학습합니다.

- Temperature는 Teacher의 출력 분포를 부드럽게 만들어 class 간 관계 정보를 드러냅니다.
- `alpha`는 KD loss와 CE loss의 비율을 조절합니다.
- KD는 큰 Teacher와 작은 Student 사이의 성능 격차가 크거나 task가 복잡할 때 더 효과적인 경우가 많습니다.
- 이후 경량화 단계는 KD Student를 기준으로 진행하면서 여러 기법이 누적될 때 크기와 정확도가 어떻게 변하는지 확인합니다.

In [ ]:
class DistillationLoss(nn.Module):
    """Teacher의 soft label과 실제 hard label을 함께 사용하는 KD loss입니다."""
    def __init__(self, temperature=2.0, alpha=0.3):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.ce = nn.CrossEntropyLoss()
        self.kl = nn.KLDivLoss(reduction="batchmean")

    def forward(self, student_logits, teacher_logits, labels):
        T = self.temperature

        ##### [실습] 실제 정답 label에 대한 일반 Cross Entropy입니다. #####
        hard_loss = self.??(student_logits, labels)

        # Teacher와 Student의 softened probability distribution 차이를 줄입니다.
        # T가 클수록 분포가 부드러워져 class 간 관계 정보가 더 드러납니다.
        soft_teacher = F.softmax(teacher_logits / T, dim=1)
        soft_student = F.log_softmax(student_logits / T, dim=1)
        soft_loss = self.kl(soft_student, soft_teacher) * (T ** 2)

        # alpha가 클수록 Teacher의 soft label을 더 강하게 따릅니다.
        return self.alpha * soft_loss + (1.0 - self.alpha) * hard_loss


def train_kd_epoch(student, teacher, loader, optimizer, criterion, device):
    ##### [실습] Teacher는 eval모드로 설정하고 Student만 KD loss로 train 합니다. #####
    student.??()
    teacher.??()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="KD Train", leave=False):
        ##### [실습] 입력 이미지와 정답 label을 학습 device(CPU/GPU)로 이동합니다. #####
        images, labels = ??, ??

        ##### [실습] Teacher 모델은 학습하지 않으므로 gradient 계산을 비활성화한 뒤, Teacher의 출력값을 얻어옵니다. #####
        with torch.no_grad():
            teacher_outputs = ??

        ##### [실습] Student 모델에 입력 이미지를 전달하여 출력값을 얻어옵니다. #####
        student_outputs = ??

        ##### [실습] Student 출력, Teacher 출력, 정답 label을 이용하여 KD loss를 계산합니다. #####
        loss = ??

        ##### [실습] 이전 step에서 계산된 gradient를 초기화합니다. #####
        ??

        ##### [실습] Backpropagation을 수행하여 gradient를 계산합니다. #####
        ??

        ##### [실습] Optimizer를 이용하여 Student 모델의 파라미터를 업데이트합니다. #####
        ??

        ##### [실습] 현재 batch의 loss를 누적하여 epoch 평균 loss를 계산합니다. #####
        ??

        ##### [실습] Student 모델의 예측 클래스를 구합니다. #####
        ??

        ##### [실습] 올바르게 예측한 샘플 수를 누적합니다. #####
        ??

        ##### [실습] 처리한 전체 샘플 수를 누적합니다. #####
        ??

    return total_loss / total, 100.0 * correct / total

In [ ]:
##### [실습] KD를 적용할 StudentCNN을 만듭니다. #####
student_kd = ??().to(train_device)
kd_optimizer = optim.Adam(student_kd.parameters(), lr=KD_LR)
kd_loss = DistillationLoss(temperature=KD_TEMPERATURE, alpha=KD_ALPHA)

for epoch in range(1, STUDENT_EPOCHS + 1):
    train_loss, train_acc = train_kd_epoch(student_kd, teacher_model, train_loader, kd_optimizer, kd_loss, train_device)
    test_acc = evaluate(student_kd, test_loader, train_device)
    print(f"Epoch {epoch}/{STUDENT_EPOCHS} | kd_loss={train_loss:.4f} | train_acc={train_acc:.2f}% | test_acc={test_acc:.2f}%")

_ = record_result("KD", student_kd, train_device, "KD를 처음부터 적용해 학습한 Student 모델입니다.")

# 6. Weight Decomposition 적용

**tensorly**의 `parafac` 함수를 사용하여 Student 모델의 큰 weight를 저차원(low-rank) 형태로 근사합니다.

Student의 `fc1` layer는 `Linear(1568 → 128)` 구조를 가지며 많은 파라미터를 사용합니다. CP 분해를 적용하면 이 weight를 더 작은 크기의 factor들로 표현할 수 있습니다.

실습에서는 분해 결과를 이용하여 기존 layer를 다음과 같이 두 개의 layer로 교체합니다.

```text
Linear(1568 → 128)
        ↓
Linear(1568 → rank)
Linear(rank → 128)
```

이렇게 하면 파라미터 수를 줄여 모델을 더 가볍게 만들 수 있습니다.

이 셀에서 배울 내용은 다음과 같습니다.

- `tensorly.decomposition.parafac()`를 이용해 weight를 분해하는 방법
- 분해된 결과를 새로운 Linear layer의 weight로 사용하는 방법
- rank 값에 따른 압축률과 성능의 trade-off

> 일반적으로 rank가 작을수록 모델 크기는 줄어들지만, 원래 weight를 덜 정확하게 근사하게 됩니다.

## 6-1. Weight Decomposition 관련 함수 정의

In [ ]:
def cp_decompose_weight_with_tensorly(weight, rank):
    """tensorly.parafac으로 2D weight matrix를 CP 분해합니다."""
    # tensorly 연산은 CPU tensor로 수행해 Colab/GPU 환경 차이에 따른 오류 가능성을 줄입니다.
    weight_cpu = weight.detach().cpu()

    ###### [실습] parafac()으로 CP 분해를 수행합니다. ######
    # rank가 작을수록 압축률은 높지만 근사 오차가 커질 수 있습니다.
    cp_factors = ??(
        weight_cpu,
        rank=rank,
        n_iter_max=100,
        tol=1e-6,
        random_state=seed,
    )

    # tensorly 0.8.1에서는 CPTensor 형태로 weights/factors에 접근할 수 있습니다.
    try:
        cp_weights = cp_factors.weights
        factor_matrices = cp_factors.factors
    except AttributeError:
        cp_weights, factor_matrices = cp_factors

    # CP weight가 None이면 각 rank component의 가중치를 1로 간주합니다.
    if cp_weights is None:
        cp_weights = torch.ones(rank)

    # 2D matrix W(out_features, in_features)를 분해하면 factor도 2개가 나옵니다.
    out_factor, in_factor = factor_matrices
    return cp_weights, out_factor, in_factor, cp_factors


def reconstruct_weight_from_cp(cp_weights, out_factor, in_factor):
    """CP factor로부터 원래 Linear weight 모양의 행렬을 재구성합니다."""
    return (out_factor * cp_weights.unsqueeze(0)) @ in_factor.T

## 6-2. Weight Decomposition 관련 모델 정의

In [ ]:
class DecomposedStudentCNN(nn.Module):
    """Student의 fc1을 CP factor 기반의 두 Linear layer로 치환한 모델입니다."""
    def __init__(self, base_model, rank=32):
        super().__init__()

        # convolution 부분은 KD Student에서 학습된 weight를 그대로 가져옵니다.
        self.conv1 = copy.deepcopy(base_model.conv1)
        self.conv2 = copy.deepcopy(base_model.conv2)
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()

        old_fc1 = base_model.fc1
        old_fc2 = base_model.fc2

        ##### [실습] fc1 weight를 tensorly CP decomposition으로 분해합니다. #####
        # 위에서 정의한 cp_decompose_weight_with_tensorly 함수를 사용합니다.
        cp_weights, out_factor, in_factor, self.cp_factors = ??(old_fc1.weight, rank)

        # reconstruction MSE는 원래 weight와 CP 근사 weight의 차이를 보여줍니다.
        reconstructed = reconstruct_weight_from_cp(cp_weights, out_factor, in_factor)
        mse = torch.mean((old_fc1.weight.detach().cpu() - reconstructed) ** 2).item()
        print(f"CP reconstruction MSE: {mse:.8f}")
        print(f"Original fc1 weight shape: {tuple(old_fc1.weight.shape)}")
        print(f"CP factor shapes: out={tuple(out_factor.shape)}, in={tuple(in_factor.shape)}, weights={tuple(cp_weights.shape)}")

        # W ~= out_factor @ diag(cp_weights) @ in_factor.T
        # Linear는 y = x @ weight.T + bias 형태로 계산하므로 다음처럼 두 layer로 나눌 수 있습니다.
        # 1) fc1_a: x @ in_factor
        # 2) fc1_b: (x @ in_factor) @ (out_factor * cp_weights).T + bias
        self.fc1_a = nn.Linear(old_fc1.in_features, rank, bias=False)
        self.fc1_b = nn.Linear(rank, old_fc1.out_features, bias=True)
        self.fc2 = copy.deepcopy(old_fc2)

        self.fc1_a.weight.data.copy_(in_factor.T)
        self.fc1_b.weight.data.copy_(out_factor * cp_weights.unsqueeze(0))
        self.fc1_b.bias.data.copy_(old_fc1.bias.detach().cpu())

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.flatten(x)
        x = F.relu(self.fc1_b(self.fc1_a(x)))
        return self.fc2(x)


rank = 32
student_decomp = DecomposedStudentCNN(student_kd.cpu(), rank=rank).to(train_device)
student_kd.to(train_device)

_ = record_result("KD > Decomp", student_decomp, train_device, f"tensorly.parafac CP rank={rank}로 fc1을 분해했습니다.")

## 6-3. Decomposition 후 짧은 Fine-tuning
분해로 생긴 근사 오차를 줄이기 위해 짧게 fine-tuning합니다.

- 구조를 바꾸는 경량화는 fine-tuning을 함께 사용할 때 안정적입니다.
- 실제 경량화 파이프라인에서도 compression 후 fine-tuning을 반복적으로 수행합니다.

In [ ]:
# Decomposition은 근사 오차를 만들 수 있으므로 짧은 fine-tuning으로 성능을 회복합니다.
decomp_optimizer = optim.Adam(student_decomp.parameters(), lr=0.0005)

for epoch in range(1, FINETUNE_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(student_decomp, train_loader, decomp_optimizer, ce_loss, train_device)
    test_acc = evaluate(student_decomp, test_loader, train_device)
    print(f"Fine-tune {epoch}/{FINETUNE_EPOCHS} | loss={train_loss:.4f} | train_acc={train_acc:.2f}% | test_acc={test_acc:.2f}%")

_ = record_result("KD > Decomp > FT", student_decomp, train_device, "분해 후 fine-tuning을 수행했습니다.")

# 7. Pruning 적용
Decomposition까지 적용한 모델에 L1 unstructured pruning을 적용합니다.

- L1 pruning은 절댓값이 작은 weight를 0으로 만듭니다.

In [ ]:
def apply_global_l1_pruning(model, amount=0.35):
    """Conv/Linear weight 전체를 대상으로 L1 기준 unstructured pruning을 적용합니다."""
    parameters_to_prune = []

    # Conv2d와 Linear layer의 weight만 pruning 대상으로 삼습니다.
    # bias까지 pruning하면 작은 모델에서는 정확도 손실이 커질 수 있어 제외합니다.
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            parameters_to_prune.append((module, "weight"))

    ##### [실습] global_unstructured pruning을 합니다. #####
    # global_unstructured는 모든 대상 layer의 weight를 한데 모아 작은 값부터 제거합니다.
    prune.??(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=amount,
    )

    # pruning re-parametrization을 제거하여 weight_orig/weight_mask 대신 실제 weight에 0을 남깁니다.
    for module, name in parameters_to_prune:
        prune.remove(module, name)

    return model


def make_zero_masks(model):
    """현재 0인 weight 위치를 기억합니다. Pruning 후 fine-tuning에서 0이 다시 살아나는 것을 막기 위해 사용합니다."""
    return {name: (param.detach() == 0) for name, param in model.named_parameters()}


def apply_zero_masks(model, masks):
    """mask가 True인 위치를 다시 0으로 만들어 pruning sparsity를 유지합니다."""
    with torch.no_grad():
        for name, param in model.named_parameters():
            if name in masks:
                param.masked_fill_(masks[name].to(param.device), 0.0)


student_pruned = copy.deepcopy(student_decomp).to(train_device)
##### [실습] pruning 비율을 0.35로 설정합니다. #####
pruning_amount = ??
student_pruned = apply_global_l1_pruning(student_pruned, amount=pruning_amount)

# 이 mask를 유지하지 않으면 fine-tuning 중 pruned weight가 다시 non-zero가 됩니다.
pruning_masks = make_zero_masks(student_pruned)

_ = record_result("KD > Decomp > FT > Pruning", student_pruned, train_device, f"전체 Conv/Linear weight의 {int(pruning_amount * 100)}%를 pruning했습니다.")

## 7-1. Pruning 후 짧은 Fine-tuning
Pruning으로 손실된 성능을 일부 회복하기 위해 짧게 fine-tuning합니다.

- pruning은 정확도 손실을 만들 수 있으므로 fine-tuning이 중요합니다.
- pruning 비율을 높이면 sparsity는 증가하지만 정확도 하락 위험도 커집니다.
- fine-tuning 중 pruned weight가 다시 살아나지 않도록 `pruning_masks`를 계속 적용해야 sparsity가 유지됩니다.

In [ ]:
def train_one_epoch_with_pruning_masks(model, loader, optimizer, criterion, device, masks):
    """Pruned weight가 다시 살아나지 않도록 mask를 유지하면서 fine-tuning합니다."""
    ##### [실습] 모델을 학습모드로 설정합니다. #####
    model.??()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Masked FT", leave=False):
        ##### [실습] 입력 이미지와 정답 label을 학습 device(CPU/GPU)로 이동합니다. #####
        images, labels = ??, ??

        ##### [실습] 모델에 입력 이미지를 전달하여 출력값을 얻어옵니다. #####
        outputs = ??

        ##### [실습] 모델의 출력값과 정답 label을 이용하여 loss를 계산합니다. #####
        loss = ??

        ##### [실습] 이전 step에서 계산된 gradient를 초기화합니다. #####
        ??

        ##### [실습] Backpropagation을 수행하여 gradient를 계산합니다. #####
        ??

        ##### [실습] Optimizer를 이용하여 모델의 파라미터를 업데이트합니다. #####
        ??

        ##### [실습] Pruning된 weight가 다시 살아나지 않도록 mask를 재적용합니다. #####
        ??

        ##### [실습] 현재 batch의 loss를 누적하여 epoch 평균 loss를 계산합니다. #####
        ??

        ##### [실습] 모델의 예측 클래스를 구합니다. #####
        ??

        ##### [실습] 올바르게 예측한 샘플 수를 누적합니다. #####
        ??

        ##### [실습] 처리한 전체 샘플 수를 누적합니다. #####
        ??

    return total_loss / total, 100.0 * correct / total


# Pruning 후 fine-tuning은 반드시 pruning mask를 유지해야 sparsity가 보존됩니다.
prune_optimizer = optim.Adam(student_pruned.parameters(), lr=0.0005)

for epoch in range(1, FINETUNE_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch_with_pruning_masks(
        student_pruned,
        train_loader,
        prune_optimizer,
        ce_loss,
        train_device,
        pruning_masks,
    )

    ##### [실습] evaluate 함수를 이용하여 test accuracy를 계산합니다. #####
    test_acc = ??(student_pruned, test_loader, train_device)
    print(f"Fine-tune {epoch}/{FINETUNE_EPOCHS} | loss={train_loss:.4f} | train_acc={train_acc:.2f}% | test_acc={test_acc:.2f}%")

_ = record_result("KD > Decomp > FT > Pruning > FT", student_pruned, train_device, "Pruning mask를 유지하며 보정 학습을 수행했습니다.")

# 8. Quantization 적용
마지막 단계로 dynamic quantization을 적용합니다. 이번 실습에서는 Linear layer를 INT8로 바꿉니다.

- quantization은 일반적으로 학습이 끝난 뒤 배포 직전에 적용합니다.
- dynamic quantization은 사용하기 쉽고 CPU에서 Linear layer 중심 모델의 저장 크기를 줄이는 데 효과적입니다.
- Conv layer는 그대로 FP32이므로 모든 layer가 INT8이 되는 것은 아닙니다.

In [ ]:
# Dynamic Quantization은 학습이 끝난 모델의 일부 layer를 더 가볍게 바꾸는 후처리 기법입니다.
# 여기서는 nn.Linear layer를 INT8 형태로 변환하여 CPU 추론 시 모델 크기를 줄입니다.
import warnings

student_quantized = copy.deepcopy(student_pruned).to(cpu_device).eval()
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=DeprecationWarning)
    ##### [실습] torch.quantization.quantize_dynamic 함수를 사용하여 양자화합니다. #####
    student_quantized = ??(
        student_quantized,
        {nn.Linear},
        dtype=torch.qint8,
    )

# 양자화된 Linear layer의 weight는 내부적으로 packed 형태로 저장됩니다.
# 따라서 일반적인 model.parameters() 기준의 파라미터 수와 sparsity는 이전 단계와 공정하게 비교하기 어렵습니다.
# 이 단계에서는 Params/Non-zero/Sparsity 대신 모델 크기(size_mb)와 정확도(accuracy)를 중심으로 확인합니다.
_ = record_result("KD > Decomp > FT > Pruning > FT > Quant", student_quantized, cpu_device, "Linear layer에 dynamic INT8 quantization을 적용했습니다.")

# 9. 전체 결과 비교
지금까지 누적 적용한 경량화 단계의 결과를 표로 비교합니다.

- 경량화는 정확도, 크기, sparsity, latency 사이의 trade-off를 보는 과정입니다.
- 어떤 기법이 항상 좋은 것이 아니라 목표 하드웨어와 목표 지표에 따라 선택 순서가 달라질 수 있습니다.

In [ ]:
print("Stage".ljust(36), "Acc(%)".rjust(8), "Size(MB)".rjust(10), "Params".rjust(12), "Non-zero".rjust(12), "Sparse(%)".rjust(10), "CPU Latency(ms/img)".rjust(20))
print("=" * 114)
for row in results:
    print(
        row["stage"].ljust(36),
        f"{row['accuracy']:.2f}".rjust(8),
        f"{row['size_mb']:.3f}".rjust(10),
        format_count(row["total_params"]).rjust(12),
        format_count(row["nonzero_params"]).rjust(12),
        format_float(row["sparsity"]).rjust(10),
        f"{row['latency_ms']:.4f}".rjust(20),
    )

## 9-1. 결과 해석 가이드
실습 결과를 다음 관점으로 해석해보세요.

1. **KD**
    - KD는 모델을 직접 작게 만들지는 않지만, Teacher의 soft label을 통해 Student의 학습 신호를 보완할 수 있습니다.
    - 다만 KD가 항상 정확도를 올리는 것은 아닙니다. 쉬운 task이거나 Teacher와 Student 성능 차이가 작으면 Teacher가 전달할 추가 정보가 제한적일 수 있습니다.


2. **Weight Decomposition**
    - `tensorly.parafac()`으로 얻은 CP factor를 실제 두 개의 Linear layer로 바꾸었기 때문에 `total_params`와 `size_mb`가 줄어드는지 확인할 수 있습니다.
    - rank를 낮추면 더 작아지지만 reconstruction MSE가 커지고 정확도가 떨어질 수 있습니다.
    - decomposition 직후 정확도가 조금 떨어졌다가 fine-tuning 후 회복되는 흐름이 자연스럽습니다.

3. **Pruning**
    - `sparsity`가 증가하는지 확인합니다.
    - pruning 후 fine-tuning을 할 때 mask를 유지하지 않으면 0으로 만든 weight가 optimizer update로 다시 살아납니다. 이 노트북은 `pruning_masks`를 유지하여 sparsity가 보존되도록 했습니다.

4. **Quantization**
    - CPU용 INT8 Linear layer로 저장 크기가 줄어드는지 확인합니다.
    - Dynamic quantization된 Linear layer는 packed parameter로 저장되어 `model.parameters()` 기준 parameter 수를 FP32 모델과 공정하게 비교하기 어렵습니다. 그래서 이 노트북에서는 quantized 모델의 `Params`, `Non-zero`, `Sparsity`를 `N/A`로 표시하고, `size_mb`와 accuracy를 중심으로 해석합니다.
    - 이 노트북의 latency는 공정한 비교를 위해 모두 CPU 기준으로 측정하며, warmup 이후 여러 batch의 중앙값을 사용합니다. 그래도 Colab CPU 상태, batch size, quantized operator 지원 여부에 따라 속도는 변동될 수 있습니다.

5. **실제 경량화 순서**
    - 일반적인 추천 흐름은 `학습된 큰 모델 준비 -> KD로 작은 모델 학습 -> 구조적 축소/decomposition -> pruning + fine-tuning -> quantization`입니다.
    - quantization은 마지막 배포 단계에 두는 경우가 많습니다. 앞 단계에서 weight가 계속 바뀌면 quantization calibration 또는 변환을 다시 해야 하기 때문입니다.